# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [13]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [14]:
# Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [15]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [16]:
import chromadb

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")

@tool
def retrieve_game(query: str) -> str:
    """
    Semantic search: Finds most relevant results in the vector DB.
    args:
    - query: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=5,
        include=["documents", "metadatas", "distances"],
    )
    docs = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        docs.append(
            f"Platform: {meta.get('Platform', 'N/A')}, "
            f"Name: {meta.get('Name', 'N/A')}, "
            f"YearOfRelease: {meta.get('YearOfRelease', 'N/A')}, "
            f"Description: {doc}"
        )
    return "\n---\n".join(docs)

#### Evaluate Retrieval Tool

In [17]:
import re
from pydantic import BaseModel, Field
from lib.parsers import PydanticOutputParser

class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")

evaluation_parser = PydanticOutputParser(model_class=EvaluationReport)

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    evaluation_llm = LLM(model="gpt-4o-mini", temperature=0.0)
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Query: {question}\n\n"
        f"Retrieved Documents:\n{retrieved_docs}\n\n"
        "Respond ONLY with valid JSON matching this schema:\n"
        '{"useful": true/false, "description": "..."}'
    )
    response = evaluation_llm.invoke(prompt)
    # Strip markdown code fences if present
    content = response.content
    content = re.sub(r"^```(?:json)?\s*\n?", "", content.strip())
    content = re.sub(r"\n?```\s*$", "", content.strip())
    response.content = content
    report = evaluation_parser.parse(response)
    return f"Useful: {report.useful}\nDescription: {report.description}"

#### Game Web Search Tool

In [18]:
from tavily import TavilyClient

@tool
def game_web_search(question: str) -> str:
    """
    Web search: Searches the web for information about the game industry.
    args:
    - question: a question about the game industry.
    """
    tavily = TavilyClient(api_key=TAVILY_API_KEY)
    response = tavily.search(query=question, max_results=5)
    results = []
    for r in response.get("results", []):
        results.append(f"Title: {r['title']}\nURL: {r['url']}\nContent: {r['content']}")
    return "\n---\n".join(results) if results else "No results found."

### Agent

In [19]:
instructions = """You are UdaPlay, an AI Research Agent specialized in the video game industry.

Your workflow for answering questions:
1. First, use retrieve_game to search the internal vector database for relevant information.
2. Then, use evaluate_retrieval to assess whether the retrieved documents are sufficient to answer the question.
3. If the evaluation says the documents are useful, answer the question based on the retrieved data.
4. If the evaluation says the documents are NOT useful, use game_web_search to find the answer on the web.
5. Always provide a clear, concise, and well-structured answer.

Important guidelines:
- Always start by searching the internal database before going to the web.
- Cite the source of your information (database or web).
- If you cannot find the answer from either source, say so honestly.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3,
)

In [20]:
# Invoke the agent with test questions

questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
]

# Reset memory so previous runs don't leak cached answers into context
agent.memory = type(agent.memory)()

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")

    # Track previous message count so we only print new messages
    last_run = agent.memory.get_last_object("default")
    prev_count = len(last_run.get_final_state()["messages"]) if last_run else 0

    run = agent.invoke(q)
    final_state = run.get_final_state()

    # Only iterate over messages added during this run
    for msg in final_state["messages"][prev_count:]:
        if isinstance(msg, AIMessage):
            if msg.tool_calls:
                calls = ", ".join(tc.function.name for tc in msg.tool_calls)
                print(f"Thought: Calling tool(s): {calls}")
                if msg.content:
                    print(f"  Reasoning: {msg.content}")
            elif msg.content:
                print(f"Answer: {msg.content}")
        elif isinstance(msg, ToolMessage) and msg.content:
            print(f"  Tool [{msg.name}]: {msg.content[:200]}...")
    print()


Q: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Thought: Calling tool(s): retrieve_game
  Tool [retrieve_game]: "Platform: Game Boy Color, Name: Pok\u00e9mon Gold and Silver, YearOfRelease: 1999, Description: [Game Boy Color] Pok\u00e9mon Gold and Silver (1999) - Second-generation Pok\u00e9mon games introducing...
Thought: Calling tool(s): evaluate_retrieval
  Tool [evaluate_retrieval]: "Useful: True\nDescription: The retrieved document provides the name of the games, 'Pok\u00e9mon Gold and Silver', along with the year of release, which is 1999. This directly answers the query regard...
Answer: Pokémon Gold and Silver were released in 1999 for t

### (Optional) Advanced

In [10]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes